# Sensitivity (impact) analysis for ReMo3D

This notebook produces 2D contour plots of the Fréchet sensitivity kernel in the (r, z) cross-section. Each plot answers the question **"how much does each area of the formation contribute to this measurement?"**

Two methods are exposed by `remo3d.sensitivity`:

* **Born / analytical** — closed-form kernel using full-space Green's functions in a *homogeneous* background ρ₀. Fast, runs in a fraction of a second. The kernel shape is exact for a homogeneous medium and a good approximation for mildly heterogeneous formations.
* **Perturbation / finite-difference** — perturb ρ in each cell, re-run `Model.compute_synthetic_logs`, take ΔRₐ / Δ(lnρ). Honours the actual layered + invaded model but costs **one FEM solve per cell** (very slow).

Convention: the kernel is the integrand of
$$\int\!\!\int S(r,z)\,dr\,dz \;\approx\; R_a^{\text{homog}} \;=\; \rho_0,$$
so each plotted value represents the contribution per unit cross-section area (the axisymmetric Jacobian $2\pi r$ is already included).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from remo3d import plot_sensitivity, analytical_sensitivity, perturbation_sensitivity
from remo3d.sensitivity import _selftest

## Example A — short normal in a homogeneous half-space

A short normal tool `N0.4M0.1A` (AM = 0.1 m, AN = 0.5 m) measured at z = 5 m in a uniform 10 Ω·m formation. Expect a classic near-tool sensitivity pattern concentrated around the M and A electrodes.

In [ ]:
depth = 5.0
rho = 10.0

# Single uniform layer that extends well above and below the measurement point
formation_homo = np.array([[depth - 50.0, depth + 50.0, np.nan, np.nan, rho]])
borehole_homo = np.array([[depth - 50.0, 0.1, rho], [depth + 50.0, 0.1, rho]])

ax = plot_sensitivity(
    tool="N0.4M0.1A", depth=depth,
    formation_model=formation_homo, borehole_model=borehole_homo,
    method="born",
    r_lim=(-1.5, 1.5), z_lim=(depth - 2.5, depth + 2.5),
    n_r=200, n_z=200,
    rho_background=rho,
)
plt.show()

## Example B — long lateral across a thin resistive bed

A long lateral `B5.7A0.4M` (under `force_single_electrode_configuration` this is equivalent to a single-current configuration with AM = 0.4 m and AN = 5.7 m) probes much deeper into the formation. The contour highlights the asymmetric, far-reaching response.

In [ ]:
# Thin resistive bed at 14–16 m embedded in a 5 Ω·m background
formation_thinbed = np.array([
    [0.0, 14.0, np.nan, np.nan, 5.0],
    [14.0, 16.0, np.nan, np.nan, 50.0],
    [16.0, 30.0, np.nan, np.nan, 5.0],
])
borehole_thinbed = np.array([[0.0, 0.108, 1.1], [30.0, 0.108, 1.1]])

ax = plot_sensitivity(
    tool="B5.7A0.4M", depth=15.0,
    formation_model=formation_thinbed, borehole_model=borehole_thinbed,
    method="born",
    r_lim=(-8.0, 8.0), z_lim=(8.0, 22.0),
    n_r=240, n_z=240,
)
plt.show()

## Example C — three tools at the same depth on the layered + invaded model

Reuses the formation/borehole files from `Forward.ipynb` to compare how different tools probe the same depth. The first two short normals see mostly the invaded zone; the long lateral reaches into the undisturbed formation.

In [ ]:
formation_file = "./Input/Ex1/Formation.txt"
borehole_file = "./Input/Ex1/Borehole.txt"
depth_compare = 5.0  # inside the invaded layer (3.05–8.35 m, FZ_radius = 0.3 m, FZ_value = 3 Ω·m, UZ = 18 Ω·m)

tools_compare = ["N0.4M0.1A", "N2.0M0.5A", "B5.7A0.4M"]

fig, axes = plt.subplots(1, 3, figsize=(18, 8), facecolor="white")
for ax, tool in zip(axes, tools_compare):
    plot_sensitivity(
        tool=tool, depth=depth_compare,
        formation_model=formation_file, borehole_model=borehole_file,
        method="born",
        r_lim=(-1.0, 1.0), z_lim=(2.0, 8.5),
        n_r=200, n_z=200,
        ax=ax,
        colorbar=False,
    )
fig.colorbar(axes[-1].collections[0], ax=axes, location="bottom", pad=0.08,
             shrink=0.6, label="∂R_a / ∂(ln ρ)  [Ω·m]")
plt.show()

## Verification

Two checks:

1. **Homogeneous integral test** — for a uniform ρ₀, the analytical kernel integrates to ρ₀ (within ≈3 % on a finite grid).
2. **Born vs. perturbation correlation** — run the perturbation finite-difference on a coarse 6×12 grid and compare against the analytical kernel sampled on the same grid. The two should correlate strongly (r > 0.9 for a well-resolved homogeneous medium).

The perturbation cell is *slow* (it triggers one FEM solve per cell). Skip it unless you want to validate the kernel.

In [ ]:
rel_err = _selftest(tool="N0.4M0.1A", depth=5.0, rho=10.0,
                    r_max=50.0, z_half_range=50.0, n_r=400, n_z=400)
print("Relative error of ∫∫ S dr dz vs. ρ₀ = {:.3%}".format(rel_err))

In [ ]:
# WARNING: ~72 FEM solves — take a coffee break. Comment this cell out if you only need the contour plots.

depth = 5.0
rho = 10.0
formation_homo = np.array([[depth - 20.0, depth + 20.0, np.nan, np.nan, rho]])
borehole_homo = np.array([[depth - 20.0, 0.1, rho], [depth + 20.0, 0.1, rho]])

r_coarse = np.linspace(0.1, 1.0, 6)
z_coarse = np.linspace(depth - 1.5, depth + 1.5, 12)

S_born, _, _ = analytical_sensitivity(
    "N0.4M0.1A", depth, formation_homo, borehole_homo,
    r_grid=r_coarse, z_grid=z_coarse, rho_background=rho)

S_pert = perturbation_sensitivity(
    "N0.4M0.1A", depth, formation_homo, borehole_homo,
    r_grid=r_coarse, z_grid=z_coarse,
    perturbation=0.05, cpu_workers=4, domain_radius=20, batch_size=1,
    verbose=False,
)

mask = np.isfinite(S_born) & np.isfinite(S_pert)
corr = np.corrcoef(S_born[mask], S_pert[mask])[0, 1]
print("Born vs perturbation correlation: {:.3f}".format(corr))

fig, axes = plt.subplots(1, 2, figsize=(10, 5), facecolor="white")
vmax = float(np.nanmax(np.abs(np.concatenate([S_born[mask], S_pert[mask]]))))
for ax, S, label in zip(axes, [S_born, S_pert], ["Born", "Perturbation"]):
    im = ax.pcolormesh(r_coarse, z_coarse, S, cmap="RdBu_r", vmin=-vmax, vmax=vmax, shading="auto")
    ax.invert_yaxis()
    ax.set_title(label)
    ax.set_xlabel("r [m]")
    ax.set_ylabel("z [m]")
fig.colorbar(im, ax=axes.tolist(), location="bottom", pad=0.12, shrink=0.7)
plt.show()